In [0]:
SELECT *
FROM read_files(
  '/Volumes/workspace/default/ibb_raw/2021-yl-istanbul-deniz-iskeleleri-yolcu-saylar.csv',
  format => 'csv',
  header => true,
  sep => ',',
  inferColumnTypes => false
)
LIMIT 5;

yil,ay,otorite_adi,istasyon_adi,tekil_yolcu_sayisi,toplam_yolculuk_sayisi,_rescued_data
2021,1,IBB TOPLU ULASIM HIZMETLERI MUDURLUGU,İstinye,1923,4408,null
2021,1,TURYOL - S.S. TURIZM VE YOLCU DENIZ TASIYICILAR KOOPERATIFI,Karaköy,29145,80531,null
2021,1,S.S. ISTANBUL KARTAL DENIZ YOL YUK VE TURZ TAS MOTORLU TAS KOOP,Büyükada,6380,18489,null
2021,1,S.S. MAVI MARMARA DEN. YOL. TURZ. MOT. TASIY. KOOP.,Bostancı 08 04 2021 Kod Deişti,13802,31328,null
2021,1,IBB TOPLU ULASIM HIZMETLERI MUDURLUGU,Kanlıca,666,1317,null


In [0]:
SELECT
  yil,
  COUNT(*) AS kayit_sayisi,
  COUNT(DISTINCT ay) AS ay_sayisi,
  SUM(CAST(toplam_yolculuk_sayisi AS BIGINT)) AS toplam_yolculuk
FROM read_files(
  '/Volumes/workspace/default/ibb_raw/202[1-4]-yl-istanbul-deniz-iskeleleri-yolcu-saylar.csv',
  format => 'csv',
  header => true,
  sep => ',',
  inferColumnTypes => false
)
GROUP BY yil
ORDER BY yil;

yil,kayit_sayisi,ay_sayisi,toplam_yolculuk
2021,979,12,47445145
2022,978,12,71572152
2023,1000,12,72087043
2024,970,12,70283867


In [0]:
CREATE OR REPLACE TABLE workspace.default.ibb_deniz_iskeleleri_bronze_2021_2024
USING DELTA AS
SELECT
  yil,
  ay,
  otorite_adi,
  istasyon_adi,
  tekil_yolcu_sayisi,
  toplam_yolculuk_sayisi,
  _metadata.file_path AS kaynak_dosya
FROM read_files(
  '/Volumes/workspace/default/ibb_raw/202[1-4]-yl-istanbul-deniz-iskeleleri-yolcu-saylar.csv',
  format => 'csv',
  header => true,
  sep => ',',
  inferColumnTypes => false
);

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  yil,
  COUNT(*) AS kayit_sayisi,
  COUNT(DISTINCT ay) AS ay_sayisi,
  COUNT(DISTINCT kaynak_dosya) AS dosya_sayisi
FROM workspace.default.ibb_deniz_iskeleleri_bronze_2021_2024
GROUP BY yil
ORDER BY yil;

yil,kayit_sayisi,ay_sayisi,dosya_sayisi
2021,979,12,1
2022,978,12,1
2023,1000,12,1
2024,970,12,1


In [0]:
CREATE OR REPLACE TABLE workspace.default.ibb_deniz_iskeleleri_silver_2021_2025
USING DELTA AS

SELECT
  CAST(yil AS INT) AS yil,
  CAST(ay AS INT) AS ay,
  otorite_adi,
  NULLIF(TRIM(istasyon_adi), '') AS istasyon_adi,
  CAST(tekil_yolcu_sayisi AS BIGINT) AS tekil_yolcu_sayisi,
  CAST(toplam_yolculuk_sayisi AS BIGINT) AS yolcu_sayisi,
  kaynak_dosya
FROM workspace.default.ibb_deniz_iskeleleri_bronze_2021_2024

UNION ALL

SELECT
  CAST(yil AS INT) AS yil,
  CAST(ay AS INT) AS ay,
  otorite_adi,
  NULLIF(TRIM(istasyon_adi), '') AS istasyon_adi,
  CAST(tekil_yolcu_sayisi AS BIGINT) AS tekil_yolcu_sayisi,
  CAST(yolcu_sayisi AS BIGINT) AS yolcu_sayisi,
  kaynak_dosya
FROM workspace.default.ibb_deniz_iskeleleri_bronze;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  yil,
  COUNT(*) AS kayit_sayisi,
  COUNT(DISTINCT ay) AS ay_sayisi,
  COUNT_IF(istasyon_adi IS NULL) AS bos_istasyon_sayisi,
  SUM(yolcu_sayisi) AS toplam_yolculuk
FROM workspace.default.ibb_deniz_iskeleleri_silver_2021_2025
GROUP BY yil
ORDER BY yil;

yil,kayit_sayisi,ay_sayisi,bos_istasyon_sayisi,toplam_yolculuk
2021,979,12,35,47445145
2022,978,12,33,71572152
2023,1000,12,43,72087043
2024,970,12,41,70283867
2025,958,12,29,70513663


In [0]:
SELECT
  yil,
  ay,
  otorite_adi,
  istasyon_adi,
  COUNT(*) AS tekrar_sayisi
FROM workspace.default.ibb_deniz_iskeleleri_silver_2021_2025
GROUP BY yil, ay, otorite_adi, istasyon_adi
HAVING COUNT(*) > 1
ORDER BY tekrar_sayisi DESC;

yil,ay,otorite_adi,istasyon_adi,tekrar_sayisi


In [0]:
CREATE OR REPLACE TABLE workspace.default.ibb_deniz_iskeleleri_gold_aylik_2021_2025
USING DELTA AS
SELECT
  yil,
  ay,
  SUM(yolcu_sayisi) AS yolcu_sayisi_toplami,
  COUNT(*) AS kaynak_satir_sayisi,
  COUNT_IF(istasyon_adi IS NULL) AS istasyonu_bilinmeyen_satir
FROM workspace.default.ibb_deniz_iskeleleri_silver_2021_2025
GROUP BY yil, ay;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  yil,
  COUNT(*) AS ay_sayisi,
  SUM(kaynak_satir_sayisi) AS kaynak_satir_toplami,
  SUM(yolcu_sayisi_toplami) AS yillik_yolculuk
FROM workspace.default.ibb_deniz_iskeleleri_gold_aylik_2021_2025
GROUP BY yil
ORDER BY yil;

yil,ay_sayisi,kaynak_satir_toplami,yillik_yolculuk
2021,12,979,47445145
2022,12,978,71572152
2023,12,1000,72087043
2024,12,970,70283867
2025,12,958,70513663


In [0]:
WITH yillik AS (
  SELECT
    yil,
    SUM(yolcu_sayisi_toplami) AS yolculuk
  FROM workspace.default.ibb_deniz_iskeleleri_gold_aylik_2021_2025
  GROUP BY yil
),
karsilastirma AS (
  SELECT
    yil,
    yolculuk,
    LAG(yolculuk) OVER (ORDER BY yil) AS onceki_yil
  FROM yillik
)
SELECT
  yil,
  yolculuk,
  onceki_yil,
  yolculuk - onceki_yil AS yillik_fark,
  ROUND(
    100.0 * (yolculuk - onceki_yil) / NULLIF(onceki_yil, 0),
    2
  ) AS yillik_degisim_yuzde
FROM karsilastirma
ORDER BY yil;

yil,yolculuk,onceki_yil,yillik_fark,yillik_degisim_yuzde
2021,47445145,null,null,null
2022,71572152,47445145,24127007,50.85
2023,72087043,71572152,514891,0.72
2024,70283867,72087043,-1803176,-2.50
2025,70513663,70283867,229796,0.33


In [0]:
CREATE OR REPLACE TABLE workspace.default.ibb_deniz_iskeleleri_gold_yillik_2021_2025
USING DELTA AS
WITH yillik AS (
  SELECT
    yil,
    SUM(yolcu_sayisi_toplami) AS yolculuk
  FROM workspace.default.ibb_deniz_iskeleleri_gold_aylik_2021_2025
  GROUP BY yil
),
karsilastirma AS (
  SELECT
    yil,
    yolculuk,
    LAG(yolculuk) OVER (ORDER BY yil) AS onceki_yil
  FROM yillik
)
SELECT
  yil,
  yolculuk,
  onceki_yil,
  yolculuk - onceki_yil AS yillik_fark,
  ROUND(
    100.0 * (yolculuk - onceki_yil) / NULLIF(onceki_yil, 0),
    2
  ) AS yillik_degisim_yuzde
FROM karsilastirma;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  ay,
  yil,
  yolcu_sayisi_toplami
FROM workspace.default.ibb_deniz_iskeleleri_gold_aylik_2021_2025
ORDER BY ay, yil;

ay,yil,yolcu_sayisi_toplami
1,2021,1834347
1,2022,3839861
1,2023,5751954
1,2024,4685515
1,2025,5539415
2,2021,2073893
2,2022,4397608
2,2023,4263709
2,2024,5486501
2,2025,3827759


Databricks visualization. Run in Databricks to view.

In [0]:
SELECT
  g25.ay,
  g24.yolcu_sayisi_toplami AS yolculuk_2024,
  g25.yolcu_sayisi_toplami AS yolculuk_2025,
  g25.yolcu_sayisi_toplami - g24.yolcu_sayisi_toplami AS fark,
  ROUND(
    100.0 * (g25.yolcu_sayisi_toplami - g24.yolcu_sayisi_toplami)
    / NULLIF(g24.yolcu_sayisi_toplami, 0),
    2
  ) AS degisim_yuzde
FROM workspace.default.ibb_deniz_iskeleleri_gold_aylik_2021_2025 AS g24
JOIN workspace.default.ibb_deniz_iskeleleri_gold_aylik_2021_2025 AS g25
  ON g24.ay = g25.ay
WHERE g24.yil = 2024
  AND g25.yil = 2025
ORDER BY g25.ay;

ay,yolculuk_2024,yolculuk_2025,fark,degisim_yuzde
1,4685515,5539415,853900,18.22
2,5486501,3827759,-1658742,-30.23
3,5433395,5308609,-124786,-2.30
4,6784560,5855155,-929405,-13.70
5,6568891,6389220,-179671,-2.74
6,6883997,7003143,119146,1.73
7,6681785,6710006,28221,0.42
8,6714157,6804211,90054,1.34
9,5479075,5796372,317297,5.79
10,5958822,6139505,180683,3.03


In [0]:
SELECT
  yil,
  ay,
  COUNT(DISTINCT otorite_adi) AS otorite_sayisi,
  COUNT(*) AS kayit_sayisi
FROM workspace.default.ibb_deniz_iskeleleri_silver_2021_2025
WHERE yil IN (2024, 2025)
GROUP BY yil, ay
ORDER BY ay, yil;

yil,ay,otorite_sayisi,kayit_sayisi
2024,1,8,81
2025,1,8,78
2024,2,8,81
2025,2,8,78
2024,3,8,80
2025,3,8,81
2024,4,8,81
2025,4,8,78
2024,5,8,84
2025,5,8,79


In [0]:
WITH otoriteler_2024 AS (
  SELECT DISTINCT ay, otorite_adi
  FROM workspace.default.ibb_deniz_iskeleleri_silver_2021_2025
  WHERE yil = 2024
),
otoriteler_2025 AS (
  SELECT DISTINCT ay, otorite_adi
  FROM workspace.default.ibb_deniz_iskeleleri_silver_2021_2025
  WHERE yil = 2025
)
SELECT
  a.ay,
  a.otorite_adi
FROM otoriteler_2024 AS a
LEFT JOIN otoriteler_2025 AS b
  ON a.ay = b.ay
 AND a.otorite_adi = b.otorite_adi
WHERE b.otorite_adi IS NULL
ORDER BY a.ay;

ay,otorite_adi
10,ISTANBUL SEHIR HATLARI TUR. SAN. VE TIC. AS.
11,ISTANBUL SEHIR HATLARI TUR. SAN. VE TIC. AS.
12,ISTANBUL SEHIR HATLARI TUR. SAN. VE TIC. AS.


In [0]:
CREATE OR REPLACE TABLE workspace.default.ibb_deniz_iskeleleri_gold_aylik_otorite_2021_2025
USING DELTA AS
SELECT
  yil,
  ay,
  otorite_adi,
  SUM(yolcu_sayisi) AS yolcu_sayisi_toplami,
  COUNT(*) AS kaynak_satir_sayisi
FROM workspace.default.ibb_deniz_iskeleleri_silver_2021_2025
GROUP BY yil, ay, otorite_adi;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  yil,
  COUNT(*) AS ay_otorite_satiri,
  COUNT(DISTINCT ay) AS ay_sayisi,
  SUM(kaynak_satir_sayisi) AS kaynak_satir_toplami,
  SUM(yolcu_sayisi_toplami) AS yillik_yolculuk
FROM workspace.default.ibb_deniz_iskeleleri_gold_aylik_otorite_2021_2025
GROUP BY yil
ORDER BY yil;

yil,ay_otorite_satiri,ay_sayisi,kaynak_satir_toplami,yillik_yolculuk
2021,96,12,979,47445145
2022,96,12,978,71572152
2023,100,12,1000,72087043
2024,96,12,970,70283867
2025,93,12,958,70513663


In [0]:
CREATE OR REPLACE TABLE workspace.default.ibb_deniz_iskeleleri_gold_karsilastirma_2024_2025
USING DELTA AS
WITH otorite_sayilari AS (
  SELECT
    yil,
    ay,
    COUNT(*) AS otorite_sayisi
  FROM workspace.default.ibb_deniz_iskeleleri_gold_aylik_otorite_2021_2025
  WHERE yil IN (2024, 2025)
  GROUP BY yil, ay
)
SELECT
  eski.ay,
  eski.yolcu_sayisi_toplami AS yolculuk_2024,
  yeni.yolcu_sayisi_toplami AS yolculuk_2025,
  yeni.yolcu_sayisi_toplami - eski.yolcu_sayisi_toplami AS fark,
  ROUND(
    100.0 * (yeni.yolcu_sayisi_toplami - eski.yolcu_sayisi_toplami)
    / NULLIF(eski.yolcu_sayisi_toplami, 0),
    2
  ) AS degisim_yuzde,
  otorite_2024.otorite_sayisi AS otorite_sayisi_2024,
  otorite_2025.otorite_sayisi AS otorite_sayisi_2025
FROM workspace.default.ibb_deniz_iskeleleri_gold_aylik_2021_2025 AS eski
JOIN workspace.default.ibb_deniz_iskeleleri_gold_aylik_2021_2025 AS yeni
  ON eski.ay = yeni.ay
JOIN otorite_sayilari AS otorite_2024
  ON otorite_2024.yil = 2024 AND otorite_2024.ay = eski.ay
JOIN otorite_sayilari AS otorite_2025
  ON otorite_2025.yil = 2025 AND otorite_2025.ay = yeni.ay
WHERE eski.yil = 2024
  AND yeni.yil = 2025;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
  ay,
  yolculuk_2024,
  yolculuk_2025,
  fark,
  degisim_yuzde,
  otorite_sayisi_2024,
  otorite_sayisi_2025
FROM workspace.default.ibb_deniz_iskeleleri_gold_karsilastirma_2024_2025
ORDER BY ay;

ay,yolculuk_2024,yolculuk_2025,fark,degisim_yuzde,otorite_sayisi_2024,otorite_sayisi_2025
1,4685515,5539415,853900,18.22,8,8
2,5486501,3827759,-1658742,-30.23,8,8
3,5433395,5308609,-124786,-2.30,8,8
4,6784560,5855155,-929405,-13.70,8,8
5,6568891,6389220,-179671,-2.74,8,8
6,6883997,7003143,119146,1.73,8,8
7,6681785,6710006,28221,0.42,8,8
8,6714157,6804211,90054,1.34,8,8
9,5479075,5796372,317297,5.79,8,8
10,5958822,6139505,180683,3.03,8,7


Databricks visualization. Run in Databricks to view.

**Veri kapsamı notu:** Ekim–Aralık 2025 kaynak verisinde `ISTANBUL SEHIR HATLARI TUR. SAN. VE TIC. AS.` için kayıt bulunmuyor. Bu aylardaki 2024–2025 farkları yayımlanan kayıtların farkıdır; eksik kaydı sıfır yolculuk olarak yorumlamıyoruz.

### Proje özeti

İBB deniz iskeleleri yolculuk verilerinin 2021–2025 dönemindeki 4.885 kaynak satırı incelendi. Farklı yıllardaki CSV şemaları Silver katmanında ortak alanlara dönüştürüldü; Gold katmanında aylık, yıllık ve otorite bazlı özetler oluşturuldu. Her yıl 12 ay içeriyor ve Gold toplamları Silver toplamlarıyla uyuşuyor.

2024–2025 karşılaştırmasında yayımlanan yıllık yolculuk toplamı 70.283.867'den 70.513.663'e çıktı (+229.796; yaklaşık %0,33). Ekim–Aralık 2025'te bir otoriteye ait kayıt bulunmadığından yıllık ve bu aylara ilişkin farklar kaynak kapsamı dikkate alınarak yorumlanmalıdır.